### Automated Stacking Ensemble Classification Pipeline with Cross-Validation and Multi-Seed Evaluation

In [33]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, StackingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    accuracy_score, precision_score, recall_score
)

BASE_INPUT_DIR = ""

# FUNCTION TO RUN PIPELINE PER FOLDER
def run_classification1_pipeline(folder_path):
    dataset_name = os.path.basename(folder_path)
    print(f"\n{'='*80}\n Processing Folder: {dataset_name}\n{'='*80}")

    try:
        # -------------------------------------------------------------
        # Load Data
        # -------------------------------------------------------------
        X_train_scaled = pd.read_csv(os.path.join(folder_path, f"hERG_train_fp_features_scaled.csv"))
        y_train = pd.read_csv(os.path.join(folder_path, f"hERG_train_fp_features_raw.csv")).Label

        X_val_scaled = pd.read_csv(os.path.join(folder_path, f"hERG_valid_fp_features_scaled.csv"))
        y_val = pd.read_csv(os.path.join(folder_path, f"hERG_valid_fp_features_raw.csv")).Label

        X_test_scaled = pd.read_csv(os.path.join(folder_path, f"hERG_test_fp_features_scaled.csv"))
        y_test = pd.read_csv(os.path.join(folder_path, f"hERG_test_fp_features_raw.csv")).Label

        # -------------------------------------------------------------
        # Save input shapes
        # -------------------------------------------------------------
        shapes_df = pd.DataFrame({
            'Split': ['Train', 'Valid', 'Test'],
            'X_shape': [str(X_train_scaled.shape), str(X_val_scaled.shape), str(X_test_scaled.shape)],
            'y_shape': [str(y_train.shape), str(y_val.shape), str(y_test.shape)]
        })
        save_path_output = os.path.join(folder_path, "Outputs")
        os.makedirs(save_path_output, exist_ok=True)
        shapes_df.to_csv(os.path.join(save_path_output, "data_shapes.csv"), index=False)

        # -------------------------------------------------------------
        # Clean variable naming
        # -------------------------------------------------------------
        X_train_fp_clean = X_train_scaled
        X_valid_fp_clean = X_val_scaled
        X_test_fp_clean  = X_test_scaled
        y_train_fp_clean = y_train
        y_valid_fp_clean = y_val
        y_test_fp_clean  = y_test

        all_results = []

        # -------------------------------------------------------------
        # Run multiple random seeds
        # -------------------------------------------------------------
        for seed in [1, 2, 3, 4, 5]:
            print(f"\n=== Training Run with seed={seed} ===")

            base_models = [
                ('et', ExtraTreesClassifier(random_state=seed, n_jobs=-1)),
                ('rf', RandomForestClassifier(random_state=seed, n_jobs=-1)),
                ('lgbm', LGBMClassifier(device='cpu', random_state=seed,
                                        force_col_wise=True, verbose=-1)),
                ('xgb', XGBClassifier(tree_method='hist', device='cuda',
                                      eval_metric='logloss', random_state=seed))
            ]

            meta_learner = LogisticRegression(solver='liblinear', random_state=seed)
            stacking_ensemble = StackingClassifier(
                estimators=base_models,
                final_estimator=meta_learner,
                cv=10,
                n_jobs=-1
            )

            # Combine train + val for final fit
            X_train_full = pd.concat([X_train_fp_clean, X_valid_fp_clean], axis=0, ignore_index=True)
            y_train_full = np.concatenate([y_train_fp_clean, y_valid_fp_clean])

            stacking_ensemble.fit(X_train_full, y_train_full)

            # Predict on test
            y_pred = stacking_ensemble.predict(X_test_fp_clean)
            y_proba = stacking_ensemble.predict_proba(X_test_fp_clean)[:, 1]

            # Metrics
            auroc = roc_auc_score(y_test_fp_clean, y_proba)
            auprc = average_precision_score(y_test_fp_clean, y_proba)
            acc   = accuracy_score(y_test_fp_clean, y_pred)
            prec  = precision_score(y_test_fp_clean, y_pred)
            rec   = recall_score(y_test_fp_clean, y_pred)

            # Save per-run predictions
            pred_df = pd.DataFrame({
                'y_true': y_test_fp_clean,
                'y_pred': y_pred,
                'y_proba': y_proba
            })
            pred_file = os.path.join(save_path_output, f"Classification_predictions_seed{seed}.csv")
            pred_df.to_csv(pred_file, index=False)

            # Save model
            model_file = os.path.join(save_path_output, f"Classification_model_seed{seed}.pkl")
            joblib.dump(stacking_ensemble, model_file)

            # Store results
            all_results.append({
                'Seed': seed,
                'AUROC': auroc,
                'AUPRC': auprc,
                'Accuracy': acc,
                'Precision': prec,
                'Recall': rec,
                'Model_File': model_file,
                'Prediction_File': pred_file
            })

            print(f"Seed {seed} -> AUROC {auroc:.4f}, AUPRC {auprc:.4f}, "
                  f"ACC {acc:.4f}, PREC {prec:.4f}, REC {rec:.4f}")

        # -------------------------------------------------------------
        # Save aggregated results
        # -------------------------------------------------------------
        results_df = pd.DataFrame(all_results)
        results_df.to_csv(os.path.join(save_path_output, "Classification_all_results.csv"), index=False)

        summary = results_df[['AUROC','AUPRC','Accuracy','Precision','Recall']].agg(['mean','std']).T
        summary['Mean ± Std'] = summary.apply(lambda x: f"{x['mean']:.4f} ± {x['std']:.4f}", axis=1)
        summary.to_csv(os.path.join(save_path_output, "Classification_summary.csv"))

        print("\n=== FINAL SUMMARY ===")
        print(summary[['Mean ± Std']])

    except Exception as e:
        print(f"❌ Error processing {dataset_name}: {e}")

# RUN FOR ALL SUBFOLDERS
for subfolder in sorted(os.listdir(BASE_INPUT_DIR)):
    folder_path = os.path.join(BASE_INPUT_DIR, subfolder)
    if os.path.isdir(folder_path):
        run_classification1_pipeline(folder_path)



 Processing Folder: 01_02_TDC_selfies_variance

=== Training Run with seed=1 ===
Seed 1 -> AUROC 0.7106, AUPRC 0.8489, ACC 0.7710, PREC 0.8148, REC 0.8980

=== Training Run with seed=2 ===
Seed 2 -> AUROC 0.7174, AUPRC 0.8461, ACC 0.7710, PREC 0.8148, REC 0.8980

=== Training Run with seed=3 ===
Seed 3 -> AUROC 0.6979, AUPRC 0.8347, ACC 0.7786, PREC 0.8165, REC 0.9082

=== Training Run with seed=4 ===
Seed 4 -> AUROC 0.7038, AUPRC 0.8357, ACC 0.7634, PREC 0.8131, REC 0.8878

=== Training Run with seed=5 ===
Seed 5 -> AUROC 0.7183, AUPRC 0.8466, ACC 0.7634, PREC 0.8131, REC 0.8878

=== FINAL SUMMARY ===
                Mean ± Std
AUROC      0.7096 ± 0.0088
AUPRC      0.8424 ± 0.0067
Accuracy   0.7695 ± 0.0064
Precision  0.8145 ± 0.0014
Recall     0.8959 ± 0.0085

 Processing Folder: 01_03_TDC_rdkit_variance

=== Training Run with seed=1 ===
Seed 1 -> AUROC 0.8227, AUPRC 0.9250, ACC 0.8397, PREC 0.8598, REC 0.9388

=== Training Run with seed=2 ===
Seed 2 -> AUROC 0.8174, AUPRC 0.9162, A

C:\Users\omidm\anaconda3\envs\myenv310-2\lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Seed 3 -> AUROC 0.8199, AUPRC 0.9188, ACC 0.8244, PREC 0.8571, REC 0.9184

=== Training Run with seed=4 ===
Seed 4 -> AUROC 0.8177, AUPRC 0.9212, ACC 0.8321, PREC 0.8585, REC 0.9286

=== Training Run with seed=5 ===
Seed 5 -> AUROC 0.8152, AUPRC 0.9216, ACC 0.8168, PREC 0.8558, REC 0.9082

=== FINAL SUMMARY ===
                Mean ± Std
AUROC      0.8247 ± 0.0100
AUPRC      0.9246 ± 0.0057
Accuracy   0.8244 ± 0.0054
Precision  0.8571 ± 0.0010
Recall     0.9184 ± 0.0072

 Processing Folder: 06_02_TDC_morgan_avalon_erg_selfies_rdkit_tfidf_variance

=== Training Run with seed=1 ===
Seed 1 -> AUROC 0.8364, AUPRC 0.9348, ACC 0.8397, PREC 0.8598, REC 0.9388

=== Training Run with seed=2 ===
Seed 2 -> AUROC 0.8327, AUPRC 0.9305, ACC 0.8321, PREC 0.8585, REC 0.9286

=== Training Run with seed=3 ===
Seed 3 -> AUROC 0.8299, AUPRC 0.9286, ACC 0.8244, PREC 0.8571, REC 0.9184

=== Training Run with seed=4 ===
Seed 4 -> AUROC 0.8336, AUPRC 0.9353, ACC 0.8321, PREC 0.8585, REC 0.9286

=== Training R